SANTIAGO DIE MOREJÓN - 23/03/2026 - IA - Entregable 1 - Envío a Kaggle

# Notebook 4: Envío a Kaggle

En el notebook de comparativa concluimos que Random Forest con 5 descriptores fue el mejor modelo con un accuracy de 0.747. En el notebook adicional optimizamos los hiperparámetros con `RandomizedSearchCV` y creamos nuevos descriptores a partir de HR y BP.

## 1. Carga de datos y librerías

Cargamos los datos desde el repositorio y preparamos las herramientas necesarias.

In [1]:
import numpy as np
import pandas as pd

In [2]:
traindf = pd.read_csv('/kaggle/input/competitions/ia-2526/train.csv')
testdf = pd.read_csv("/kaggle/input/competitions/ia-2526/test.csv")

## 2. Preparación de los datos

In [4]:
traindf['SL'] = np.log1p(traindf['SL'])
traindf['CIRCULATION'] = np.log1p(traindf['CIRCULATION'])

testdf['SL'] = np.log1p(testdf['SL'])
testdf['CIRCULATION'] = np.log1p(testdf['CIRCULATION'])

In [ ]:
traindf['HR_BP_ratio'] = traindf['HR'] / (traindf['BP'] + 1e-6)
traindf['HR_BP_product'] = traindf['HR'] * traindf['BP']
traindf['HR_BP_diff'] = traindf['HR'] - traindf['BP']

testdf['HR_BP_ratio'] = testdf['HR'] / (testdf['BP'] + 1e-6)
testdf['HR_BP_product'] = testdf['HR'] * testdf['BP']
testdf['HR_BP_diff'] = testdf['HR'] - testdf['BP']

In [ ]:
cols = ['SL', 'EEG', 'BP', 'HR', 'CIRCULATION', 'HR_BP_ratio', 'HR_BP_product', 'HR_BP_diff']

X_train = traindf[cols].values
y_train = traindf['ACTIVITY'].values

X_test = testdf[cols].values

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}')

## 3. Entrenamiento del modelo final

En el notebook de optimización encontramos con `RandomizedSearchCV` que los mejores hiperparámetros para Random Forest son `n_estimators=321`, `max_depth=30`, `min_samples_split=5`.

Usamos directamente esos valores y entrenamos sobre todo el conjunto de entrenamiento para aprovechar al máximo los datos.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
modelo_final = RandomForestClassifier(n_estimators=321, max_depth=30, min_samples_split=5, random_state=42)
modelo_final.fit(X_train, y_train)

## 4. Predicciones y generación del CSV de envío

Realizamos las predicciones sobre el conjunto de test y generamos el fichero CSV con el formato que se pide.

In [ ]:
predicciones = modelo_final.predict(X_test)

In [ ]:
soldf = pd.DataFrame([(i+1,int(pred)) for i, pred in enumerate(predicciones)])
soldf.columns = ['Id','Prediction']
soldf.head(10)

In [ ]:
soldf.to_csv("submission.csv",sep=',',index=False)